# 03 · Modelado: YOLO11-s, fast-plate-ocr y regla de decisión

**SIRAP** · Proyecto Integrador en IA (MIAR0545) · Eduardo Alejandro Ceballos Jijón · Christian Xavier Valle Maridueña

Entrena YOLO11-s con el protocolo del proyecto. Como el modelo ya está entrenado (preset `completo`), **no reentrena**: muestra el registro y las curvas de entrenamiento (3.1 a 3.3). Carga fast-plate-ocr (3.4) y define la lista de placas autorizadas y la regla de decisión (3.5 y 3.6). Usa la normalización de 2.1.

> Este notebook es un extracto de `SIRAP_backend.ipynb` (el notebook completo, que es la fuente de verdad). Las celdas y sus salidas son las mismas y conservan la numeración de secciones del notebook completo; la sección 0 se repite para que corra por sí solo en Colab.


## 0. Preparación del entorno

Esta sección deja lista la sesión de Colab: comprueba la GPU, instala las librerías, lee las llaves de API, monta Google Drive, define todas las rutas de entrada y fija la configuración del experimento. Hay que correrla completa al empezar cada sesión; no entrena ni evalúa nada.

### GPU
`nvidia-smi` muestra la GPU asignada. Si falla, ve a `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `GPU T4` y guarda. Sin GPU el notebook funciona, pero entrenar y evaluar el detector se vuelve muy lento (el OCR corre en la CPU en cualquier caso).

In [ ]:
!nvidia-smi

### Dependencias
Instala solo las librerías que Colab no trae, en las versiones con las que se probó el notebook, para que los resultados se puedan reproducir:

- `ultralytics==8.4.177`: entrenamiento, validación e inferencia de YOLO11.
- `trackers==2.6.1`: OC-SORT, el seguidor de placas del pipeline de video.
- `supervision==0.30.9`: detecciones, anotación de video y mAP por tamaño de placa.
- `fast-plate-ocr==1.1.0` y `onnxruntime==1.31.0`: el OCR de placas (`cct-s-v2-global-model`) y el motor que lo corre en la CPU. fast-plate-ocr se instala sin sus dependencias (`--no-deps`): pide `opencv-python-headless`, que podría chocar con el `opencv-python` de Colab (dos paquetes para el mismo `cv2`), y las demás (`numpy`, `pyyaml`, `rich` y `tqdm`) ya vienen en Colab.
- `roboflow==1.7.1`: descarga del dataset.
- `Levenshtein`: distancia entre placas (regla de decisión y CER).

`torch`, `torchvision`, `opencv`, `pandas` y `matplotlib` ya vienen en Colab y no se tocan. Si al terminar Colab pide **reiniciar la sesión**, reinicia y sigue desde la celda siguiente (no hace falta reinstalar).

La celda de importaciones, más abajo, carga todas las librerías y muestra la GPU y las versiones instaladas.

In [ ]:
!pip install -q "ultralytics==8.4.177" "trackers==2.6.1" "supervision==0.30.9" "roboflow==1.7.1" Levenshtein "onnxruntime==1.31.0"
!pip install -q --no-deps "fast-plate-ocr==1.1.0"  # sin dependencias: usa el OpenCV que ya trae Colab

### Llaves de API (*Secrets* de Colab)

Las llaves no se escriben en el notebook: la función `get_secret` las lee del panel 🔑 **Secrets** de Colab (fuera de Colab, de variables de entorno).

| Secret | Para qué | ¿Obligatorio? |
|---|---|---|
| `ROBOFLOW_API_KEY` | Descargar el dataset de placas desde Roboflow (Roboflow → `Settings` → `API`) | Sí, con `DATASET_SOURCE = "roboflow"` |
| `CAMARA_URL` | Dirección de la cámara para el modo en vivo: la del túnel de Cloudflare (`https://….trycloudflare.com`) | No (sección 9) |
| `CAMARA_TOKEN` | Token que imprime `puente_camara.py` en la PC de la cámara | Con el puente (sección 9) |
| `CF_ACCESS_CLIENT_ID` y `CF_ACCESS_CLIENT_SECRET` | Token de servicio, si el túnel está protegido con Cloudflare Access | No (sección 9) |

En Colab: panel izquierdo → 🔑 **Secrets** → agrega cada nombre con su valor y activa el acceso desde el notebook.

> ⚠️ Una versión anterior de este notebook tenía dos llaves de Roboflow escritas en celdas, así que quedaron guardadas en ese archivo y en cualquier copia que se haya compartido. **Revócalas en Roboflow (`Settings` → `API`), genera una nueva** y usa solo el *Secret*.

In [ ]:
import os


def get_secret(name: str, required: bool = False):
    """Lee un secreto de Colab (panel 🔑) o, fuera de Colab, de una variable de entorno."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except ImportError:
        value = os.environ.get(name)
    except Exception:  # el secreto no existe o el notebook no tiene acceso
        value = None
    if required and not value:
        raise RuntimeError(f"Falta el secreto {name}: agrégalo en el panel 🔑 Secrets de Colab.")
    return value

In [ ]:
import csv
import copy
import gc
import hashlib
import json
import random
import re
import shutil
import subprocess
import time
import urllib.request
import zipfile
from importlib.metadata import version
from math import comb
from pathlib import Path

import cv2
import Levenshtein
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import supervision as sv
import torch
import yaml
from IPython.display import Image as IPyImage, Video, display
from PIL import Image, ImageDraw, ImageFont
from supervision.metrics import MeanAveragePrecision
from tqdm.auto import tqdm
from trackers import OCSORTTracker
from ultralytics import YOLO
from ultralytics.data.utils import IMG_FORMATS
from ultralytics.utils.torch_utils import get_flops, get_num_params

CUDA = torch.cuda.is_available()
DEVICE = 0 if CUDA else "cpu"
FP16 = 16 if CUDA else None  # inferencia en FP16 en la GPU (argumento `quantize` de Ultralytics 8.4)

print("GPU:", torch.cuda.get_device_name(0) if CUDA else "no disponible: todo será muy lento")
for package in ["torch", "ultralytics", "trackers", "supervision", "fast-plate-ocr", "onnxruntime"]:
    print(f"{package:<16}{version(package)}")

### Carpeta de trabajo en Google Drive
Colab borra `/content` al cerrar la sesión y entrenar el detector toma tiempo, así que todo se guarda en `MyDrive/proyecto_integrador_placas`. Si la sesión se cae a mitad de un entrenamiento, al volver a correr la celda de ese modelo **se reanuda desde la última época guardada**; y los resultados finales (sección 8) se pueden correr en otra sesión, porque lee todo desde aquí.

La celda siguiente monta Drive, define **todas** las rutas de entrada en un solo lugar y al final muestra cuáles encontró (`ok`, `vacío` o `no está`). Las que faltan no detienen el notebook: la sección que usa cada una se salta.

| Ruta en `MyDrive/proyecto_integrador_placas/` | Para qué | Sección |
|---|---|---|
| `savh_ground_truth_subset.zip` | 739 recortes de SAVH Guayaquil con la lectura de la cámara en `ground_truth.csv` | 1.3 |
| Hoja de Google **Revisión placas SAVH (142 legibles)** | Correcciones de esa referencia (se lee con tu cuenta de Google) | 2.2 |
| `revision_placas_142.csv` | Copia en CSV de la hoja, solo si no se puede leer la hoja (Archivo → Descargar → CSV) | 2.2 |
| `authorized_plates.csv` | Placas autorizadas fuera de la base de la planta (excepciones), una por línea | 3.5 y 6.3 |
| `video_camiones.mp4` | Video real del punto de control, con `USE_SAMPLE_VIDEO = False` | 6.1 |
| `video_ground_truth.csv` | Referencia anotada del video (placa e intervalo de cuadros) | 7.1 y 7.2 |
| `garita/imagenes/` y `garita/ground_truth.csv` | Imágenes completas de la garita con la placa transcrita | 7.3 |
| `dataset_placas_yolo.zip` | Dataset propio en formato YOLO, si `DATASET_SOURCE = "local"` | 1.1 |

`SAVH_Guayaquil_ANPR_Plate_Images_Dataset.zip` y `SAVH_YOLOv8CLS_Plate_Legibility_Dataset.zip.zip` también están en la carpeta, pero este notebook no los usa: el primero tiene las 2000 imágenes de las que sale el subconjunto, y el segundo es el dataset de legibilidad derivado de LPLCv2 (placas brasileñas). Los datos de vehículos y autorizaciones se conservan en archivos CSV.

**Salidas.** Cada experimento escribe en `experimentos/<dataset>/`: `runs/` (entrenamientos y validaciones de Ultralytics), `resultados/` (métricas en JSON y CSV, y registros de decisiones), `figuras/` (gráficos para el informe) y `videos/` (videos anotados). La evaluación del OCR en SAVH va a `ocr/`; la base de la planta se actualiza en su propio archivo, con una copia del original en `respaldo_bd/`. El modo en vivo (sección 9) guarda en `en_vivo/` el script del puente para la PC de la cámara y, por cada sesión, la grabación, el registro de decisiones y el resumen.

**Para retomar en una sesión nueva:** corre la sección 0 completa (no reentrena: los pesos ya están en Drive) y después las celdas que pide la sección que necesites, según la tabla de la portada (**Cómo usarlo**).

In [ ]:
USE_DRIVE = True  # guarda todo en Google Drive

WORK_DIR = Path("/content/proyecto_integrador_placas")
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK_DIR = Path("/content/drive/MyDrive/proyecto_integrador_placas")
    except ImportError:
        print("Fuera de Colab: se usa una carpeta local.")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", WORK_DIR)

# ===================== Archivos de entrada (todas las rutas en un solo lugar) =====================
SAVH_ZIP_CANDIDATES = [WORK_DIR / "savh_ground_truth_subset.zip", Path("/content/savh_ground_truth_subset.zip")]  # sección 1.3
REVIEW_SHEET_ID = "1l79GrAYGFkTiOYxgKDT1_BEi_b-6Duo1_QqUNlcv7yM"  # hoja «Revisión placas SAVH (142 legibles)»
REVIEW_CSV_FALLBACK = WORK_DIR / "revision_placas_142.csv"        # copia en CSV de la hoja (opcional)
AUTHORIZED_LIST_PATH = WORK_DIR / "authorized_plates.csv"
REAL_VIDEO_PATH = WORK_DIR / "video_camiones.mp4"                 # video real del punto de control
VIDEO_GROUND_TRUTH_PATH = WORK_DIR / "video_ground_truth.csv"     # columnas: plate_gt, frame_start, frame_end
E2E_IMAGES_DIR = WORK_DIR / "garita" / "imagenes"                 # imágenes completas de la garita (sección 7.3)
E2E_GROUND_TRUTH_CSV = WORK_DIR / "garita" / "ground_truth.csv"   # columnas: image_path, plate_gt
LOCAL_DATASET_ZIP = WORK_DIR / "dataset_placas_yolo.zip"          # solo con DATASET_SOURCE = "local"
LIVE_DIR = WORK_DIR / "en_vivo"                                   # modo en vivo (sección 9): puente, grabaciones y registros
BRIDGE_SCRIPT_PATH = LIVE_DIR / "puente_camara.py"                # script para la PC de la cámara (celda 9.1)
LIVE_DIR.mkdir(exist_ok=True)


def input_status(path: Path) -> str:
    """Estado de una entrada: «ok», «vacío» (archivo de 0 bytes) o «no está»."""
    if not path.exists():
        return "no está"
    if path.is_file() and path.stat().st_size == 0:
        return "vacío"
    return "ok"


savh_zip_path = next((p for p in SAVH_ZIP_CANDIDATES if p.exists()), SAVH_ZIP_CANDIDATES[0])
display(pd.DataFrame(
    [{"entrada": label, "estado": input_status(path), "ruta": str(path)} for label, path in (
        ("SAVH: recortes y ground_truth.csv", savh_zip_path),
        ("Revisión de placas: copia CSV (opcional)", REVIEW_CSV_FALLBACK),
        ("Placas autorizadas (excepciones)", AUTHORIZED_LIST_PATH),        ("Video real", REAL_VIDEO_PATH),
        ("Referencia del video", VIDEO_GROUND_TRUTH_PATH),
        ("Imágenes de la garita", E2E_IMAGES_DIR),
        ("Referencia de la garita", E2E_GROUND_TRUTH_CSV),
        ("Dataset propio (YOLO)", LOCAL_DATASET_ZIP),
    )]).set_index("entrada"))
print("La hoja de revisión se lee en la sección 2.2 con tu cuenta de Google (ID " + REVIEW_SHEET_ID + ").")

### Configuración

Todos los parámetros del experimento están en la celda siguiente, agrupados por tema:

- **Dataset:** origen (`DATASET_SOURCE`) y proyecto y versión de Roboflow.
- **Detector:** pesos iniciales, `batch` y argumentos de inferencia (`infer_kwargs`, vacío: YOLO11 usa el NMS por defecto de Ultralytics). `video_imgsz` es el tamaño al que el detector analiza el video, las imágenes de la garita y la cámara en vivo: 1280 px (la resolución de la cámara); la evaluación en *val* y *test* usa siempre 640, el tamaño de entrenamiento. `CONFIGS` tiene una entrada por configuración del detector, con el nombre que aparece en la base y en las tablas (y en el video, con `SHOW_VIDEO_TITLE = True`); la recorren las secciones 6, 7 y 9.
- **Entrenamiento:** preset, épocas, fracción de los datos, tamaño de imagen, paciencia del *early stopping* y semilla.
- **Umbrales:** piso del umbral calibrado y precisión mínima para decidir sin verificación manual; los umbrales mismos se calibran en la sección 4.1.
- **Seguimiento y OCR:** cuadros para confirmar un *track*, tamaños mínimos de placa para seguirla y para leerla, modelo de fast-plate-ocr y confianza mínima del OCR.
- **Control de acceso:** formatos válidos de placa.
- **Metas del proyecto:** mAP@0.5 ≥ 0,85, EMR ≥ 80 % y ≤ 1 s por imagen, que aparecen como líneas de referencia en los gráficos.
- **Carpetas de salida y utilidades** comunes: nombre de la corrida, cargar el detector y armar sus argumentos de inferencia (`detector_kwargs`), guardar y leer resultados en JSON.

**Protocolo de entrenamiento.** Dataset y splits fijos, tamaño de imagen 640, semilla 0, *early stopping* con paciencia 10 y lote efectivo de 64 imágenes: Ultralytics acumula gradientes hasta 64 imágenes por paso (`nbs=64`), así que `batch` solo cambia la memoria usada. Con este tamaño de dataset Ultralytics elige AdamW.

**Por qué YOLO11-s.** Es un detector pequeño (≈9,4 M de parámetros y ≈21 GFLOPs a 640 px; mAP@0.5:0.95 de 47,0 en COCO según Ultralytics) que corre rápido incluso en hardware modesto y se exporta a ONNX para correr en CPU dentro de la app. Las placas son una sola clase, mucho más simple que las 80 de COCO, y en *test* supera con holgura la meta de mAP@0.5 (sección 5.1). Otros tamaños: `yolo11n.pt`, más liviano, o `yolo11m.pt` y `yolo11l.pt`, más precisos en COCO.

**NMS.** YOLO11 propone varias cajas por objeto y se queda con la mejor mediante NMS (supresión de no máximos), con el umbral de solapamiento por defecto de Ultralytics (`iou=0.7`), igual al validar y al predecir. Las latencias de la sección 5.3 incluyen el NMS, en el postproceso.

**Presets.** `rapido` entrena con una muestra aleatoria fija del 30 % de *train* y valida cada época con el 25 % de *val*, en 15 épocas: sirve para probar el flujo completo y tener números preliminares. `completo` usa todo con 40 épocas y es el que va al informe. Mira cuánto tarda la primera época antes de dejarlo corriendo: con las 2 CPU de Colab la carga de datos suele ser el cuello de botella.

**Umbrales.** Los umbrales de confianza no se fijan a mano: se calibran sobre *val* en la sección 4.1. El estudio de sensibilidad de los hiperparámetros de entrenamiento está en la sección 4.2.


In [ ]:
# ===================== Dataset (detalles en la sección 1.1) =====================
DATASET_SOURCE = "roboflow"  # "roboflow" | "local"
RF_WORKSPACE = "roboflow-universe-projects"
RF_PROJECT = "license-plate-recognition-rxg4e"
RF_VERSION = 11
DATASET_ID = f"{RF_PROJECT}-v{RF_VERSION}" if DATASET_SOURCE == "roboflow" else LOCAL_DATASET_ZIP.stem

# ===================== Detector =====================
DETECTORS = {
    "YOLO11": {
        "loader": YOLO,
        "weights": "yolo11s.pt",         # yolo11n / s / m / l / x
        "batch": 16,
        "deterministic": True,
        "infer_kwargs": {},              # NMS por defecto de Ultralytics (iou=0.7): YOLO11 no tiene cabeza sin NMS
        "video_imgsz": 1280,             # video, garita y en vivo: la resolución de la cámara (a 640 no ve placas chicas)
    },
}
MODEL_COLORS = {"YOLO11": "#2a78d6"}  # color del modelo en todos los gráficos

# ===================== Configuraciones que corren el pipeline (secciones 6, 7 y 9) =====================
# etiqueta → (modelo, cambios en la inferencia, sufijo de los resultados). Una variante de inferencia
# (otros argumentos de predict) se agregaría aquí con su propio sufijo, sin tocar los resultados principales.
CONFIGS = {name: (name, None, "") for name in DETECTORS}

# ===================== Entrenamiento =====================
TRAINING_PRESET = "completo"  # "rapido" para probar el flujo · "completo" para el informe
PRESETS = {
    "rapido":   {"epochs": 15, "train_fraction": 0.30, "val_fraction": 0.25},
    "completo": {"epochs": 40, "train_fraction": 1.00, "val_fraction": 1.00},
}
EPOCHS = PRESETS[TRAINING_PRESET]["epochs"]
TRAIN_FRACTION = PRESETS[TRAINING_PRESET]["train_fraction"]
VAL_FRACTION = PRESETS[TRAINING_PRESET]["val_fraction"]
IMGSZ = 640
PATIENCE = 10          # early stopping: épocas seguidas sin mejorar en val
SEED = 0
WORKERS = 2            # Colab tiene 2 CPU
EVAL_BATCH = 16
FORCE_RETRAIN = False  # True borra el entrenamiento guardado de este preset y lo repite desde cero

# ===================== Umbrales de confianza (se calibran en la sección 4.1) =====================
CONF_FLOOR = 0.10                 # piso para el umbral calibrado
DECISION_PRECISION_TARGET = 0.95  # precisión mínima en val para aceptar una detección sin verificación manual

# ===================== Seguimiento y OCR (secciones 3.4 y 6) =====================
MINIMUM_CONSECUTIVE_FRAMES = 10
MIN_PLATE_AREA_RATIO = 0.001  # placas más chicas (fracción del frame) ni se siguen
OCR_PLATE_AREA_RATIO = 0.002  # tamaño mínimo para leer: en video_camiones.mp4 las placas de camiones detenidos miden 0,3–0,5 % del frame
OCR_MODEL_NAME = "cct-s-v2-global-model"  # fast-plate-ocr; "cct-xs-v2-global-model" es más chico y, en SAVH, algo menos preciso
OCR_CONF_MIN = None   # confianza mínima del OCR para decidir sin verificación manual; None = no se usa. 4.3 sugiere 0.99 (el de la app)

# ===================== Control de acceso (secciones 2.1 y 3.6) =====================
# Placas ecuatorianas: 3 letras + 3 o 4 dígitos. Con la revisión de placas, las 472 legibles de SAVH siguen uno
# de esos dos formatos. Lecturas con 2 letras + 4 o 5 dígitos solo aparecen por errores de la cámara en la
# referencia (GB01825 es GBO-1825): aceptarlas haría que normalize_plate las diera por buenas sin corregirlas.
PLATE_PATTERNS = [(3, 4), (3, 3)]  # (letras, dígitos), por frecuencia
# Para decidir, guardar y mostrar, las placas válidas se escriben XXX-0000 (format_plate, sección 2.1).

# ===================== Metas del proyecto (líneas de referencia en los gráficos) =====================
TARGET_MAP50 = 0.85
TARGET_EMR = 0.80
TARGET_E2E_S = 1.0

COLOR = sv.ColorPalette.from_hex([  # un color por track en los videos anotados
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

# ===================== Carpetas de salida (separadas por dataset) =====================
EXPERIMENT_DIR = WORK_DIR / "experimentos" / DATASET_ID
RUNS_DIR = EXPERIMENT_DIR / "runs"           # entrenamientos y validaciones de Ultralytics
RESULTS_DIR = EXPERIMENT_DIR / "resultados"  # métricas por modelo (JSON / CSV) y registros de decisiones
FIGURES_DIR = EXPERIMENT_DIR / "figuras"     # gráficos (PNG) para el informe
VIDEOS_DIR = EXPERIMENT_DIR / "videos"       # videos anotados
OCR_RESULTS_DIR = WORK_DIR / "ocr"           # evaluación del OCR en SAVH (no depende del detector)
for folder in (RUNS_DIR, RESULTS_DIR, FIGURES_DIR, VIDEOS_DIR, OCR_RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Salidas de este experimento:", EXPERIMENT_DIR)


# ===================== Utilidades comunes =====================
def run_name(name: str) -> str:
    """Identificador de la corrida: modelo + pesos + preset, p. ej. 'YOLO11_yolo11s_rapido'."""
    return f"{name}_{Path(DETECTORS[name]['weights']).stem}_{TRAINING_PRESET}"


def best_weights_path(name: str) -> Path:
    """Pesos con el mejor mAP@0.5:0.95 de validación de la corrida (los que usa todo el notebook)."""
    return RUNS_DIR / run_name(name) / "weights" / "best.pt"


def trained_models() -> list:
    """Modelos de DETECTORS que ya tienen best.pt en este preset (avisa cuáles faltan)."""
    ready = [name for name in DETECTORS if best_weights_path(name).exists()]
    missing = [name for name in DETECTORS if name not in ready]
    if missing:
        print("Aviso: todavía sin entrenar en este preset:", ", ".join(missing))
    return ready


def load_detector(name: str):
    """Carga el detector entrenado (best.pt) de la corrida."""
    return DETECTORS[name]["loader"](str(best_weights_path(name)))


def detector_kwargs(name: str, override=None) -> dict:
    """Argumentos de inferencia del modelo (`infer_kwargs` en DETECTORS). `override` los cambia para una variante de
    CONFIGS; un valor None vuelve al valor por defecto de Ultralytics."""
    kwargs = {**DETECTORS[name]["infer_kwargs"], **(override or {})}
    return {key: value for key, value in kwargs.items() if value is not None}


def video_imgsz(name: str) -> int:
    """Tamaño de inferencia en el video, la garita y el modo en vivo (`video_imgsz` en DETECTORS; None = IMGSZ). La
    calibración y la evaluación en val y test (secciones 4.1 y 5) usan siempre IMGSZ, el tamaño de entrenamiento."""
    return DETECTORS[name].get("video_imgsz") or IMGSZ


def _to_json(value):
    """Convierte tipos de numpy y rutas a tipos que json.dumps sabe guardar."""
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"No se puede guardar {type(value)} en JSON")


def save_result(name: str, kind: str, data: dict) -> None:
    """Guarda un resultado del modelo en resultados/<corrida>_<kind>.json (lo leen otras secciones y la 8)."""
    (RESULTS_DIR / f"{run_name(name)}_{kind}.json").write_text(json.dumps(data, indent=2, default=_to_json))


def load_result(name: str, kind: str):
    """Lee resultados/<corrida>_<kind>.json, o None si todavía no existe."""
    path = RESULTS_DIR / f"{run_name(name)}_{kind}.json"
    return json.loads(path.read_text()) if path.exists() else None


def free_gpu() -> None:
    """Libera memoria de la GPU entre evaluaciones largas."""
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()


def to_detections(result) -> sv.Detections:
    """Resultado de Ultralytics → sv.Detections en float32 (en FP16 las cajas llegan en float16)."""
    detections = sv.Detections.from_ultralytics(result)
    detections.xyxy = detections.xyxy.astype(np.float32)
    if detections.confidence is not None:
        detections.confidence = detections.confidence.astype(np.float32)
    return detections

## 1. Datos y análisis exploratorio

El sistema usa dos conjuntos de datos con fines distintos: un dataset público de **detección** (imágenes con la caja de cada placa) para entrenar y evaluar YOLO11-s, y recortes reales de placas ecuatorianas de **SAVH Guayaquil** para evaluar la lectura del OCR. El video y las imágenes de la garita se usan en las secciones 6 y 7.

### 1.1 Dataset de detección de placas

El detector se entrena y evalúa con un dataset fijo y sus splits. Hay dos fuentes posibles:

- `DATASET_SOURCE = "roboflow"` (por defecto): **License Plate Recognition** de Roboflow Universe (`roboflow-universe-projects/license-plate-recognition-rxg4e`, versión 11: 10 125 imágenes — 7 057 *train*, 2 048 *val*, 1 020 *test* —, una clase, licencia CC BY 4.0). No es específico de Ecuador: sirve para aprender a detectar placas en general. En la celda de configuración puedes cambiar `RF_WORKSPACE`/`RF_PROJECT`/`RF_VERSION` por un proyecto propio.
- `DATASET_SOURCE = "local"`: un `.zip` propio en formato YOLO (`train/`, `valid/`, `test/` con `images/` y `labels/`, más `data.yaml`) en la carpeta de Drive, por ejemplo las imágenes de la garita anotadas en Roboflow y exportadas en formato YOLO.

**Para el informe:** lo ideal es que `test` tenga imágenes del dominio real (camiones en la garita). Un mAP alto en un test genérico no garantiza lo mismo con placas ecuatorianas.

La celda copia el dataset al disco local de Colab, reescribe `data.yaml` con rutas absolutas (el de Roboflow trae rutas `../train/images` que no siempre se resuelven bien) y, en el preset `rapido`, crea las listas con la muestra aleatoria fija. Imprime la carpeta del dataset y sus clases.


In [ ]:
# 1.1 · Dataset de detección: descarga (o .zip propio), data.yaml con rutas absolutas y muestra del preset
# Disco local de Colab: leer miles de imágenes desde Drive en cada época sería mucho más lento.
DATASET_DIR = Path("/content/datasets") / DATASET_ID

# Obtener la clave de API de Roboflow desde los Secretos de Colab
ROBOFLOW_API_KEY = get_secret("ROBOFLOW_API_KEY")

def get_dataset() -> Path:
    """Descarga o descomprime el dataset (una sola vez por sesión) y devuelve la carpeta con data.yaml."""
    if not list(DATASET_DIR.rglob("data.yaml")):
        if DATASET_SOURCE == "roboflow":
            if not ROBOFLOW_API_KEY:
                raise RuntimeError("Falta el secreto ROBOFLOW_API_KEY (sección 0).")
            from roboflow import Roboflow
            DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
            project = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(RF_WORKSPACE).project(RF_PROJECT)
            # "yolov8" es el nombre del formato de exportación de Roboflow: el mismo formato de etiquetas YOLO que usa YOLO11
            project.version(RF_VERSION).download("yolov8", location=str(DATASET_DIR), overwrite=True)
        elif DATASET_SOURCE == "local":
            with zipfile.ZipFile(LOCAL_DATASET_ZIP) as archive:
                archive.extractall(DATASET_DIR)
        else:
            raise ValueError("DATASET_SOURCE debe ser 'roboflow' o 'local'")
    return min(DATASET_DIR.rglob("data.yaml"), key=lambda p: len(p.parts)).parent


def write_data_yaml(root: Path) -> Path:
    """data.yaml con rutas absolutas y los splits que existan (Roboflow llama 'valid' al de validación)."""
    original = yaml.safe_load((root / "data.yaml").read_text())
    names = original.get("names", ["placa"])
    if isinstance(names, list):
        names = dict(enumerate(names))
    data = {"path": str(root.resolve()), "names": names}
    for key, candidates in {"train": ["train"], "val": ["valid", "val"], "test": ["test"]}.items():
        folder = next((c for c in candidates if (root / c / "images").is_dir()), None)
        if folder:
            data[key] = f"{folder}/images"
    if "test" not in data:
        print("Aviso: el dataset no trae split 'test'; se evaluará sobre val (no es lo ideal).")
        data["test"] = data["val"]
    if len(names) > 1:
        print(f"Aviso: el dataset tiene {len(names)} clases; el pipeline asume que todas son placas.")
    out = root / "data_abs.yaml"
    out.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    return out


def list_images(folder: Path) -> list:
    """Imágenes de una carpeta (extensiones que acepta Ultralytics), en orden."""
    return sorted(p for p in Path(folder).iterdir() if p.suffix[1:].lower() in IMG_FORMATS)


def write_subset_yaml(data_yaml: Path) -> Path:
    """Preset 'rapido': muestra aleatoria fija de train y val, la misma en cada corrida."""
    if TRAIN_FRACTION >= 1 and VAL_FRACTION >= 1:
        return data_yaml
    data = yaml.safe_load(data_yaml.read_text())
    root, rng = Path(data["path"]), random.Random(SEED)
    for key, fraction in (("train", TRAIN_FRACTION), ("val", VAL_FRACTION)):
        images = list_images(root / data[key])
        chosen = sorted(rng.sample(images, max(1, round(len(images) * fraction))))
        listing = root / f"{key}_{TRAINING_PRESET}.txt"
        listing.write_text("\n".join(str(p) for p in chosen) + "\n")
        data[key] = str(listing)
    out = root / f"data_{TRAINING_PRESET}.yaml"
    out.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    return out


DATASET_ROOT = get_dataset()
DATA_YAML = write_data_yaml(DATASET_ROOT)       # evaluación: val y test completos
TRAIN_DATA_YAML = write_subset_yaml(DATA_YAML)  # entrenamiento: muestra fija en el preset 'rapido'
DATA = yaml.safe_load(DATA_YAML.read_text())
print("Dataset:", DATASET_ROOT, "| clases:", DATA["names"])

## 2. Preprocesamiento

**Imágenes del detector.** El preprocesamiento de las imágenes de detección ya quedó hecho en 1.1 (copia local del dataset, `data.yaml` con rutas absolutas y muestra fija del preset `rapido`); el resto lo aplica Ultralytics al entrenar e inferir: *letterbox* a 640 px conservando la proporción y los aumentos por defecto (mosaico, cambios de color HSV, volteo horizontal, escala y traslación). Antes del OCR, cada placa se recorta con un margen de 10 px (`crop_plate`, sección 3.4).

**Texto de las placas.** Esta sección prepara lo que se compara con el OCR: la normalización de cualquier lectura al formato ecuatoriano (2.1) y la referencia corregida de SAVH (2.2).

### 2.1 Normalización de placas
El texto de un lector de placas no siempre es una placa válida. Antes de decidir el acceso hace falta encontrar la placa en el texto, corregir confusiones típicas por posición y validar el formato. fast-plate-ocr ya devuelve solo letras y dígitos, casi siempre con forma de placa: en SAVH, 398 de sus 399 lecturas de legibles ya tenían un formato válido y la normalización corrigió una sola (`GP1226` → `GPI226`). La misma normalización se aplica también a la lectura de la cámara de SAVH, que sí trae errores de formato, y a lo que escribe a mano el operador en la app.

Las placas ecuatorianas tienen **3 letras + 3 o 4 dígitos** ([El Universo](https://www.eluniverso.com/noticias/2020/06/26/nota/7885368/placas-carros-como-identificarlas/)). En la referencia de SAVH aparecen también lecturas con 2 letras + 4 o 5 dígitos, pero salen de errores de la cámara (`GB01825` es `GBO-1825`), así que solo se aceptan los dos formatos reales.

`normalize_plate` hace tres pasos:

1. **Busca la placa dentro del texto.** Corta el texto en tramos (por espacios, guiones, saltos de línea y otra puntuación), descarta la leyenda `ECUADOR` impresa sobre la placa si viene en el texto, y prueba cada tramo y las uniones de 2 o 3 tramos seguidos (`GBA 1234`).
2. **Corrige por posición.** En las 3 primeras posiciones (letras) cambia un dígito por la letra parecida (0→O, 1→I, 5→S, 6→G, 8→B) y en las siguientes (dígitos) una letra por el dígito parecido (O→0, D→0, Q→0, I→1, Z→2, B→8); `|` se lee como 1 y `$` como S. Gana el candidato que valida con menos cambios, y se aceptan como máximo 2: en SAVH, ninguna lectura que necesitó 3 era correcta.
3. **Valida el formato.** Si ningún candidato valida, la lectura queda inválida y la regla de decisión la manda a verificación manual.

**Formato para decidir, guardar y mostrar.** `format_plate` escribe cada placa válida como en Ecuador: 3 letras, guion y 4 dígitos (`GBA-1234`). A las placas antiguas de 3 dígitos les agrega un 0 al inicio de los números (`GMS-429` → `GMS-0429`), para que todas queden en el formato XXX-0000; por eso `GMS-429` y `GMS-0429` cuentan como la misma placa. Ese es el formato de la lista de autorizados, de la regla de decisión, del registro CSV de vehículos, de los registros y del video. La evaluación del OCR con SAVH compara las placas sin guion, porque mide la lectura y no el formato.

La celda termina con casos de prueba (lecturas reales de la cámara de SAVH y textos con la leyenda, separadores y símbolos) que deben pasar antes de seguir.


In [ ]:
# 2.1 · Normalización de placas al formato ecuatoriano
CONFUSIONS_TO_LETTER = {"0": "O", "1": "I", "5": "S", "6": "G", "8": "B"}  # dígito leído donde va una letra
CONFUSIONS_TO_DIGIT = {"O": "0", "D": "0", "Q": "0", "I": "1", "Z": "2", "B": "8"}  # letra leída donde va un dígito
SYMBOLS_TO_CHAR = {"|": "1", "$": "S"}  # símbolos que un lector de texto libre puede escribir en lugar de un carácter
SEPARATORS = " \t\r\n-‐‑‒–—_.·•:,;+\"'`*()[]{}<>"  # cortan el texto; cualquier otro símbolo invalida su tramo
MAX_CORRECTIONS = 2       # en SAVH, ninguna lectura que necesitó 3 cambios era correcta
PLATE_HEADER = "ECUADOR"  # leyenda impresa sobre la placa: se descarta si un lector la transcribe junto con la placa
_SEGMENT = re.compile("[^" + re.escape(SEPARATORS) + "]+")  # tramo: caracteres seguidos sin separadores


def clean_plate_text(text) -> str:
    """Mayúsculas y solo letras y dígitos: 'gba-1234' → 'GBA1234'."""
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())


def is_plate_header(segment: str) -> bool:
    """'ECUADOR' y sus lecturas aproximadas ('EQUADOR', 'ECUNDERG'): solo letras y a 3 cambios o menos."""
    return (len(segment) >= 5 and segment.isascii() and segment.isalpha()
            and Levenshtein.distance(segment, PLATE_HEADER) <= 3)


def as_plate(candidate: str, corrections: bool = True):
    """(placa, cambios) si el candidato tiene formato de placa, corrigiendo por posición si hace falta; si no, None."""
    for n_letters, n_digits in PLATE_PATTERNS:
        if len(candidate) != n_letters + n_digits:
            continue
        letters, digits = candidate[:n_letters], candidate[n_letters:]
        if corrections:
            letters = "".join(CONFUSIONS_TO_LETTER.get(c, c) for c in letters)
            digits = "".join(CONFUSIONS_TO_DIGIT.get(c, c) for c in digits)
        plate = letters + digits
        changes = sum(a != b for a, b in zip(plate, candidate))
        if changes <= MAX_CORRECTIONS and re.fullmatch(f"[A-Z]{{{n_letters}}}[0-9]{{{n_digits}}}", plate):
            return plate, changes
    return None


def find_plate(raw_text, search: bool = True, corrections: bool = True) -> tuple:
    """Busca la placa en el texto del OCR → (placa, formato válido, cambios, (inicio, fin) en el texto).

    Prueba cada tramo entre separadores y las uniones de 2 o 3 tramos seguidos ('GBA 1234'), sin la leyenda
    ECUADOR. Gana el candidato que valida con menos cambios y, a igualdad, el que aparece primero. Si ninguno
    valida, devuelve el texto limpio sin la leyenda, marcado como inválido. `search=False` y `corrections=False`
    apagan cada paso, para la ablación."""
    text = str(raw_text).upper()
    if not search:  # el texto entero como un solo candidato
        found = as_plate(clean_plate_text(text), corrections)
        return (found[0], True, found[1], None) if found else (clean_plate_text(text), False, 0, None)
    segments = [(m.start(), m.end(), "".join(SYMBOLS_TO_CHAR.get(c, c) for c in m.group()))
                for m in _SEGMENT.finditer(text)]
    segments = [s for s in segments if not is_plate_header(s[2])]
    best = None
    for i in range(len(segments)):
        for j in range(i + 1, min(i + 3, len(segments)) + 1):
            found = as_plate("".join(s[2] for s in segments[i:j]), corrections)
            if found and (best is None or found[1] < best[2]):
                best = (found[0], True, found[1], (segments[i][0], segments[j - 1][1]))
    return best or (clean_plate_text("".join(s[2] for s in segments)), False, 0, None)


def normalize_plate(raw_text: str) -> tuple[str, bool]:
    """Texto del OCR → (placa normalizada, formato válido)."""
    plate, is_valid, _, _ = find_plate(raw_text)
    return plate, is_valid


def format_plate(text) -> str:
    """Formato de Ecuador para decidir, guardar y mostrar: 3 letras, guion y 4 dígitos ('gba1234' → 'GBA-1234'). A las
    placas antiguas de 3 dígitos se les agrega un 0 al inicio de los números ('GMS-429' → 'GMS-0429'). Un texto que no
    es una placa válida queda limpio (mayúsculas, solo letras y dígitos), sin guion."""
    plate = clean_plate_text(text)
    match = re.fullmatch(r"([A-Z]{3})(\d{3,4})", plate)
    return f"{match.group(1)}-{match.group(2).zfill(4)}" if match else plate


def plate_confidence(text: str, char_probs) -> float:
    """Confianza del OCR en la placa: la menor probabilidad entre sus letras y dígitos dentro del texto
    (sin placa válida, entre todas las letras y dígitos). NaN si no hay probabilidades."""
    if char_probs is None or len(char_probs) != len(text):
        return float("nan")
    span = find_plate(text)[3]
    start, end = span if span and len(text.upper()) == len(text) else (0, len(text))
    probs = [p for c, p in zip(text[start:end], char_probs[start:end]) if c.isalnum() or c in SYMBOLS_TO_CHAR]
    return float(min(probs)) if probs else 0.0


# Casos de prueba: lecturas reales de la cámara en SAVH (revisión de placas) y textos con la leyenda ECUADOR,
# separadores y símbolos. Cada uno debe dar la placa esperada.
for raw, expected in [("GB01825", "GBO1825"), ("GS11034", "GSI1034"), ("6TK6566", "GTK6566"), ("GTE5D45", "GTE5045"),
                      ("GP1986", "GPI986"), ("gba-1234", "GBA1234"), ("ECUADOR\nGMS-429", "GMS429"),
                      ("EQUIDOR\nGTE-7040", "GTE7040"), ("ECUNDERG\nGB0-6445", "GBO6445"), ("G5V-3233", "GSV3233"),
                      ("G50-6164", "GSO6164"), ("G|G-430", "GIG430"), ("GSH+2315", "GSH2315"), ("GTT·4568", "GTT4568"),
                      ("ECUADOR GMS 429", "GMS429")]:
    normalized, is_valid = normalize_plate(raw)
    assert is_valid and normalized == expected, f"{raw!r} → {normalized} (se esperaba {expected})"
# Lecturas que deben quedar inválidas (→ verificación manual) en vez de convertirse en otra placa válida.
for raw in ["GTT-187G", "GPA-1G88", "OBX=660", "GBΦ6184", "115220", "ECUADOR", ""]:
    normalized, is_valid = normalize_plate(raw)
    assert not is_valid, f"{raw!r} no debería validar (salió {normalized})"
assert plate_confidence("ECUADOR\nGMS-429", [0.2] * 8 + [0.9, 0.8, 0.95, 0.3, 0.99, 0.7, 0.9]) == 0.7
for raw, expected in [("gba1234", "GBA-1234"), ("GBA-1234", "GBA-1234"), ("GMS429", "GMS-0429"), ("gms-429", "GMS-0429"),
                      ("PBA 123", "PBA-0123"), ("PBA-0123", "PBA-0123"), ("G8A1234", "G8A1234"), ("", "")]:
    assert format_plate(raw) == expected, f"{raw!r} → {format_plate(raw)!r} (se esperaba {expected!r})"
print("normalize_plate y format_plate: los casos de prueba pasan.")

## 3. Modelado

El sistema combina dos modelos y una regla: el detector **YOLO11-s** encuentra la placa (3.1 a 3.3), el OCR **fast-plate-ocr** la lee (3.4) y una **regla de decisión** compara la placa normalizada con la lista de autorizados (3.5 y 3.6). La justificación de cada elección está en `docs/arquitectura.md` del repositorio.

### 3.1 Entrenamiento del detector YOLO11-s

YOLO11-s se entrena en la celda 3.2. La función `train_detector` (celda siguiente) entrena con Ultralytics usando el preset elegido y guarda todo en `runs/<modelo>_<pesos>_<preset>/` dentro de la carpeta de Drive:

- **Si ya terminó** (es el caso: preset `completo`, 40 épocas), la celda lo detecta (`train_done.json`) y **no reentrena**: muestra el registro guardado. `FORCE_RETRAIN = True` lo repite desde cero.
- **Si la sesión se cae** a mitad de un entrenamiento, corre de nuevo las secciones 0 y 1.1, la celda de `train_detector` y luego la 3.2: **reanuda** desde `last.pt`. Antes comprueba que ese archivo conserve el estado del optimizador: Ultralytics lo borra al terminar, y pedirle reanudar un `last.pt` así no da error, sino que empieza en silencio un entrenamiento nuevo con valores por defecto (coco8 y 100 épocas).
- **`train_done.json`** guarda épocas corridas, mejor época, mAP@0.5:0.95 de validación, horas de entrenamiento (sumando sesiones) y memoria pico de GPU; todo entra luego en los resultados finales.

El entrenamiento es determinista (`deterministic=True`, semilla 0). Los hiperparámetros son los del preset (sección 0, «Protocolo de entrenamiento»); el estudio de sensibilidad está en la sección 4.2.


In [ ]:
# 3.1 · Función de entrenamiento (no reentrena si train_done.json ya existe)
def training_time_seconds(history: pd.DataFrame) -> float:
    """Tiempo total sumando sesiones: la columna 'time' de results.csv vuelve a 0 al reanudar."""
    t = history["time"].to_numpy(dtype=float)
    restarts = [t[i - 1] for i in range(1, len(t)) if t[i] < t[i - 1]]
    return float(sum(restarts) + (t[-1] if len(t) else 0.0))


def train_detector(name: str) -> Path:
    """Entrena (o reanuda) el detector con el preset elegido; si ya terminó, solo muestra su registro."""
    cfg = DETECTORS[name]
    run_dir = RUNS_DIR / run_name(name)
    best, last, done_flag = best_weights_path(name), run_dir / "weights" / "last.pt", run_dir / "train_done.json"

    if FORCE_RETRAIN and run_dir.exists():
        shutil.rmtree(run_dir)
    if done_flag.exists() and best.exists():
        print(f"[{name}] Ya entrenado en {run_dir}. Usa FORCE_RETRAIN = True para repetirlo.")
        display(pd.Series(json.loads(done_flag.read_text()), name=name).to_frame())
        return best

    if CUDA:
        torch.cuda.reset_peak_memory_stats()
    if last.exists():
        model = cfg["loader"](str(last))
        # Ultralytics borra el estado del optimizador de last.pt al terminar. Si se le pide reanudar un
        # last.pt así, no da error: empieza un entrenamiento NUEVO con argumentos por defecto (coco8, 100 épocas).
        if model.ckpt and model.ckpt.get("epoch", -1) >= 0 and model.ckpt.get("optimizer") is not None:
            print(f"[{name}] Reanudando desde {last}")
            model.train(resume=True)
        elif best.exists():
            print(f"[{name}] El entrenamiento ya había terminado; se registra con el best.pt guardado.")
        else:
            raise RuntimeError(f"{last} no se puede reanudar y no hay best.pt: usa FORCE_RETRAIN = True.")
    else:
        cfg["loader"](cfg["weights"]).train(
            data=str(TRAIN_DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=cfg["batch"], patience=PATIENCE,
            seed=SEED, deterministic=cfg["deterministic"], workers=WORKERS, device=DEVICE,
            project=str(RUNS_DIR), name=run_name(name), exist_ok=True, plots=True,
            **cfg["infer_kwargs"],
        )

    history = pd.read_csv(run_dir / "results.csv")
    info = {
        "modelo": name, "pesos_iniciales": cfg["weights"], "preset": TRAINING_PRESET,
        "epocas_config": EPOCHS, "epocas_corridas": int(history["epoch"].max()),
        "mejor_epoca": int(history.loc[history["metrics/mAP50-95(B)"].idxmax(), "epoch"]),
        "val_mAP50-95_mejor": float(history["metrics/mAP50-95(B)"].max()),
        "horas_entrenamiento": training_time_seconds(history) / 3600,
        "gpu_mem_pico_gb": torch.cuda.max_memory_reserved() / 1e9 if CUDA else None,
        "batch": cfg["batch"], "imgsz": IMGSZ, "train_fraction": TRAIN_FRACTION, "val_fraction": VAL_FRACTION,
    }
    done_flag.write_text(json.dumps(info, indent=2))
    free_gpu()
    display(pd.Series(info, name=name).to_frame())
    if (run_dir / "results.png").exists():
        display(IPyImage(filename=str(run_dir / "results.png"), width=900))
    return best

### 3.2 Entrenar YOLO11-s
Entrena YOLO11-s desde `yolo11s.pt` (preentrenado en COCO) con `batch=16`. La validación de cada época aplica el NMS de YOLO11, igual que en inferencia, y con esa métrica se elige `best.pt`. Como el modelo ya está entrenado, no reentrena: muestra el registro del entrenamiento y la gráfica de pérdidas y métricas por época de Ultralytics (`results.png`).


In [ ]:
# 3.2 · Entrenar YOLO11-s (ya entrenado: muestra el registro y no reentrena)
train_detector("YOLO11")

### 3.3 Registro del entrenamiento (preset «completo»)
Lee de Drive el registro del entrenamiento ya hecho (`train_done.json` y `results.csv` de Ultralytics) y dibuja las pérdidas de *train* y *val* (caja, clasificación y DFL) y las métricas de validación por época. No entrena nada: sirve para documentar el modelo que usa la app sin repetir el entrenamiento. Si las pérdidas de *val* siguen bajando al final, el modelo no estaba sobreajustado.


In [ ]:
# 3.3 · Registro del entrenamiento ya hecho (no entrena): train_done.json y results.csv guardados en Drive
train_dir = RUNS_DIR / run_name("YOLO11")
train_info_path, history_path = train_dir / "train_done.json", train_dir / "results.csv"
if not history_path.exists():
    print(f"No está {history_path}: corre 3.2 para entrenar YOLO11.")
else:
    if train_info_path.exists():
        display(pd.Series(json.loads(train_info_path.read_text()), name="YOLO11").to_frame())
    history = pd.read_csv(history_path)
    history.columns = history.columns.str.strip()
    best_row = history.loc[history["metrics/mAP50-95(B)"].idxmax()]
    print(f"{len(history)} épocas · mejor época {int(best_row['epoch'])}: mAP@0.5 = {best_row['metrics/mAP50(B)']:.3f}, "
          f"mAP@0.5:0.95 = {best_row['metrics/mAP50-95(B)']:.3f} en val")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    for split, style in (("train", "-"), ("val", "--")):
        for loss, color in (("box_loss", "#2a78d6"), ("cls_loss", "#eb6834"), ("dfl_loss", "#1baf7a")):
            axes[0].plot(history["epoch"], history[f"{split}/{loss}"], style, color=color, label=f"{split} {loss}")
    axes[0].set_title("Pérdidas por época (— train, - - val)")
    axes[0].legend(fontsize=8, ncol=2)
    axes[1].plot(history["epoch"], history["metrics/mAP50(B)"], color="#2a78d6", label="mAP@0.5")
    axes[1].plot(history["epoch"], history["metrics/mAP50-95(B)"], color="#0b2a6b", label="mAP@0.5:0.95")
    axes[1].axhline(TARGET_MAP50, color="#9aa4b2", linestyle=":", label=f"meta mAP@0.5 = {TARGET_MAP50}")
    axes[1].set_title("mAP en val por época")
    axes[1].legend(fontsize=8)
    axes[2].plot(history["epoch"], history["metrics/precision(B)"], color="#1baf7a", label="precisión")
    axes[2].plot(history["epoch"], history["metrics/recall(B)"], color="#eb6834", label="recall")
    axes[2].set_title("Precisión y recall en val por época")
    axes[2].legend(fontsize=8)
    for ax in axes:
        ax.set_xlabel("época")
        ax.grid(alpha=0.3)
    fig.suptitle(f"YOLO11-s · entrenamiento preset «{TRAINING_PRESET}» · {DATASET_ID}", x=0.01, ha="left", fontsize=12)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"entrenamiento_YOLO11_{TRAINING_PRESET}.png", dpi=200)
    plt.show()


### 3.4 OCR: fast-plate-ocr

El OCR recibe el recorte de la caja que entrega el detector. **El OCR es [fast-plate-ocr](https://github.com/ankandrew/fast-plate-ocr)**, con el modelo `cct-s-v2-global-model` (`OCR_MODEL_NAME`, en la configuración):

- Está hecho solo para placas: lleva el recorte a 128×64 px y predice un carácter (0–9 o A–Z) en cada una de sus 10 posiciones, que quedan vacías al final si la placa es más corta. No escribe separadores ni otro texto.
- Pesa 5 MB en formato ONNX y corre en la CPU con ONNX Runtime: unos 16 ms por placa con los 2 núcleos de Colab (media en SAVH; p95 20 ms), sin ocupar la GPU, que queda para el detector. Tiene licencia MIT.
- Se entrenó con placas de muchos países, pero Ecuador no está en su lista de regiones. Por eso hay que medirlo con placas ecuatorianas, que es lo que hacen las secciones 4.3 y 5.4 con SAVH. Afinarlo con placas ecuatorianas no lo mejoró (`notebooks/06_afinamiento_ocr.ipynb`).
- Da la probabilidad de cada carácter. `run_ocr_scored` la devuelve junto con el texto, y de ahí sale la confianza del OCR que se estudia en 4.3 y que se guarda en los registros de decisiones.

Las dos celdas siguientes cargan el modelo y definen las funciones de lectura (`run_ocr_scored`, `run_ocr`) y de recorte (`crop_plate`); al final lo prueban con una placa sintética.


In [ ]:
# 3.4 · Carga de fast-plate-ocr
from fast_plate_ocr import LicensePlateRecognizer

# En la CPU con ONNX Runtime: unos 16 ms por placa, y la GPU queda para el detector. En cada sesión nueva descarga el
# modelo (5 MB) de las versiones publicadas de fast-plate-ocr en GitHub.
OCR_DEVICE = "cpu"
ocr_model = LicensePlateRecognizer(OCR_MODEL_NAME, device=OCR_DEVICE)
print(f"OCR: fast-plate-ocr {OCR_MODEL_NAME} en {OCR_DEVICE} · entrada {ocr_model.config.img_width}×"
      f"{ocr_model.config.img_height} px · {ocr_model.config.max_plate_slots} posiciones · caracteres "
      f"{ocr_model.config.alphabet.replace(ocr_model.config.pad_char, '')}")

In [ ]:
# 3.4 · Funciones de lectura y recorte, y prueba con una placa sintética
def run_ocr_scored(image: Image.Image) -> tuple:
    """Corre fast-plate-ocr sobre una imagen PIL → (texto, probabilidad de cada carácter del texto).

    El modelo predice un carácter en cada posición; las posiciones vacías del final no van ni en el texto ni en las
    probabilidades. La lectura es determinista."""
    prediction = ocr_model.run(np.asarray(image.convert("RGB")), return_confidence=True)[0]
    text = prediction.plate
    char_probs = None if prediction.char_probs is None else [float(p) for p in prediction.char_probs[:len(text)]]
    return text, char_probs


def run_ocr(image: Image.Image) -> str:
    """Corre fast-plate-ocr sobre una imagen PIL y devuelve el texto."""
    return run_ocr_scored(image)[0]


def crop_plate(frame: np.ndarray, xyxy, pad: int = 10) -> Image.Image:
    """Recorte de la placa con `pad` píxeles de margen, como imagen PIL en RGB (lo que recibe el OCR)."""
    height, width = frame.shape[:2]
    box = sv.clip_boxes(sv.pad_boxes(xyxy=np.asarray(xyxy, dtype=float).reshape(1, 4), px=pad, py=pad), (width, height))
    return Image.fromarray(cv2.cvtColor(sv.crop_image(frame, box[0]), cv2.COLOR_BGR2RGB))


def synthetic_plate(text: str = "GBA-1234") -> Image.Image:
    """Placa sintética para comprobar que el OCR responde antes de correr evaluaciones largas."""
    image = Image.new("RGB", (360, 120), "white")
    draw = ImageDraw.Draw(image)
    draw.rectangle([4, 4, 355, 115], outline="black", width=4)
    try:
        font = ImageFont.load_default(size=64)
    except TypeError:  # Pillow < 10.1
        font = ImageFont.load_default()
    left, top, right, bottom = draw.textbbox((0, 0), text, font=font)
    draw.text(((360 - (right - left)) / 2 - left, (120 - (bottom - top)) / 2 - top), text, fill="black", font=font)
    return image


run_ocr(synthetic_plate())  # la primera lectura prepara ONNX Runtime y no se cuenta en el tiempo
start = time.perf_counter()
print("Prueba de OCR:", repr(run_ocr(synthetic_plate())), f"({1000 * (time.perf_counter() - start):.0f} ms)")

### 3.5 Lista de placas autorizadas

Un CSV con una placa por línea en la carpeta de Drive (`authorized_plates.csv`). Si no existe se crea vacío para que el pipeline no falle. **Este CSV es la fuente de placas autorizadas para la decisión de acceso. La lista simulada con la que se evalúa la regla en SAVH (sección 5.5) se guarda aparte, en `ocr/authorized_plates_simulada.csv`, y no reemplaza a esta. Las placas se pasan al formato XXX-0000 con `format_plate`, así que `GBA-1234`, `gba1234` y `GBA 1234` cuentan como la misma, igual que `GMS-429` y `GMS-0429`.


In [ ]:
# 3.5 · Lista de placas autorizadas. format_plate y clean_plate_text están en la normalización (2.1).


def load_authorized_plates(path: Path) -> set:
    """Placas del CSV en formato XXX-0000; si el archivo no existe, lo crea vacío."""
    if not path.exists():
        path.write_text("")
        print(f"Aviso: {path} no existía; se creó vacío. Agrega placas autorizadas, una por línea.")
    with open(path, newline="") as f:
        plates = {format_plate(row[0]) for row in csv.reader(f) if row and row[0].strip()}  # XXX-0000
    return plates - {"PLACA", "PLATE"}  # por si el CSV trae encabezado


def edit_distance_leq_1(a: str, b: str) -> bool:
    """True si las placas están a un carácter de distancia (Levenshtein ≤ 1): posible lectura de una autorizada."""
    if abs(len(a) - len(b)) > 1:
        return False
    return Levenshtein.distance(a, b) <= 1


AUTHORIZED_PLATES = load_authorized_plates(AUTHORIZED_LIST_PATH)
print(f"{len(AUTHORIZED_PLATES)} placas autorizadas en {AUTHORIZED_LIST_PATH}")

### 3.6 Regla de decisión
`decide_access` devuelve uno de tres estados y revisa las condiciones en este orden:

1. formato inválido, confianza de detección menor que el umbral de decisión del detector o, si está activada, confianza del OCR baja → `VERIFICACION_MANUAL`;
2. coincidencia exacta con una placa autorizada → `AUTORIZADO`;
3. a un carácter de una placa autorizada (distancia de Levenshtein ≤ 1) → `VERIFICACION_MANUAL`: puede ser la placa autorizada mal leída;
4. si no coincide con nada → `NO_REGISTRADO`.

El umbral de detección no es un 0,30 fijo: el detector usa su `conf_decision` calibrado en *val* (sección 4.1). La confianza del OCR solo cuenta si `OCR_CONF_MIN` tiene un valor; en este notebook queda en `None` y el estudio de la sección 4.3 sugiere 0,99, que es el que usa la app de SIRAP (`config/config.yaml`). En el pipeline de video (sección 6) la lista de autorizados incluye además las placas activas de la base de la planta, y una placa registrada pero inactiva pasa de `NO_REGISTRADO` a `VERIFICACION_MANUAL`.


In [ ]:
# 3.6 · Regla de decisión
def decide_access(normalized_text: str, is_valid: bool, det_conf: float, authorized: set, conf_min: float,
                  ocr_conf: float = 1.0, ocr_conf_min=None) -> str:
    """Regla de tres estados. `conf_min` es el umbral de decisión calibrado para el detector; `ocr_conf_min`, el de la
    confianza del OCR (None = no se usa; una confianza NaN cuenta como baja)."""
    low_ocr_conf = ocr_conf_min is not None and not ocr_conf >= ocr_conf_min
    if not is_valid or det_conf < conf_min or low_ocr_conf:
        return "VERIFICACION_MANUAL"

    if normalized_text in authorized:
        return "AUTORIZADO"

    if any(edit_distance_leq_1(normalized_text, p) for p in authorized):
        return "VERIFICACION_MANUAL"

    return "NO_REGISTRADO"